In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv("data/course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(0.0)

cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('NA')

df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Question 1

In [5]:
df.industry.mode()

0    technology
Name: industry, dtype: object

### Question 2

In [6]:
df[num_cols].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [7]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3

In [8]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = round(df_train[cat_cols].apply(mutual_info_converted_score), 2)
mi.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

### Question 4

In [9]:
# OHE
dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

In [13]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)

round((y_val == (y_pred >= 0.5)).mean(), 2)

np.float64(0.64)

### Question 5

In [43]:
all_features = list(df_train.columns)

train_dict = df_train.to_dict(orient='records')
val_dict = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = (model.predict_proba(X_val)[:, 1] >= 0.5)
orig_accuracy = (y_val == y_pred).mean()

differences = {}

for feature in all_features:
    subset = [f for f in all_features if f != feature]
    
    train_dict = df_train[subset].to_dict(orient='records')
    val_dict = df_val[subset].to_dict(orient='records')
    
    dv_sub = DictVectorizer(sparse=False)
    X_train_sub = dv_sub.fit_transform(train_dict)
    X_val_sub = dv_sub.transform(val_dict)
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    y_pred_sub = (model_sub.predict_proba(X_val_sub)[:, 1] >= 0.5)
    acc_sub = (y_val == y_pred_sub).mean()
    
    differences[feature] = orig_accuracy - acc_sub

df_diff = pd.DataFrame(differences.items(), columns=['feature', 'difference'])
df_diff.sort_values(by='difference')

,feature,difference
4,annual_income,-0.079
1,industry,0.000
3,location,0.000
2,employment_status,0.000
7,lead_score,0.001
5,number_of_courses_viewed,0.002
0,lead_source,0.003
6,interaction_count,0.044


### Question 6

In [44]:
dv = DictVectorizer(sparse=False)

train_dict = df_train.to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val.to_dict(orient='records')
X_val = dv.transform(val_dict)

c_values = [0.000001, 0.00001, 0.0001, 0.001]
scores = {}

for c in c_values:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = (model.predict_proba(X_val)[:, 1] >= 0.5)
    acc = round((y_val == y_pred).mean(), 3)
    
    scores[c] = acc
    print(f'C={c:<8}: accuracy={acc}')

C=1e-06   : accuracy=0.598
C=1e-05   : accuracy=0.598
C=0.0001  : accuracy=0.613
C=0.001   : accuracy=0.645
